# 5 — Visualización orientada a preguntas

Un gráfico no es el objetivo: es evidencia para responder una pregunta. En cada ejercicio primero escribí qué querés averiguar, luego construí la agregación y finalmente redactá una conclusión de dos o tres oraciones.

Este notebook se ejecuta manualmente **después** de que el Job termina correctamente; no forma parte del pipeline.

In [0]:
dbutils.widgets.text("student_id", "maxi_fronciani", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [0]:
%run ../common/config

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
daily = qualified_table(config, "gold_daily_sales")
risk = qualified_table(config, "gold_customer_risk")
batch = qualified_table(config, "gold_batch_summary")
for table in [daily, risk, batch]:
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá primero el Job completo."

## Ejemplo resuelto

**Pregunta:** ¿Qué categoría concentra el mayor monto vendido?

La agregación se hace con Spark. Sólo el resultado pequeño se convierte a Pandas para dibujarlo. Un gráfico de barras ordenado permite comparar categorías; un gráfico de torta haría más difícil comparar valores cercanos.

In [0]:
category_sales = (spark.table(daily)
    .groupBy("category")
    .agg(F.sum("total_amount").alias("total_amount"))
    .orderBy(F.desc("total_amount")))

plot_data = category_sales.toPandas().sort_values("total_amount")
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(plot_data["category"], plot_data["total_amount"], color="#2563eb")
ax.set_title("Monto vendido por categoría")
ax.set_xlabel("Monto total")
ax.set_ylabel("Categoría")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

leader = category_sales.first()
print(f"Respuesta: {leader.category} es la categoría con mayor monto vendido ({leader.total_amount:,.2f}).")

## Consigna 1 — Evolución temporal

**Pregunta:** ¿Qué día tuvo el mayor monto vendido y ese día también fue el de mayor cantidad de transacciones?

Construí una visualización temporal que permita comparar ambas métricas sin ocultar sus unidades. Explicá si los dos máximos coinciden y qué implica que coincidan —o no— sobre el ticket promedio. Fuente sugerida: `gold_daily_sales`.

In [0]:
diario = (spark.table(daily)
    .groupBy("sale_date")
    .agg(F.sum("total_amount").cast("double").alias("total_amount"),
         F.sum("transaction_count").alias("transaction_count"))
    .orderBy("sale_date"))

pdf = diario.toPandas()
max_monto = pdf.loc[pdf["total_amount"].idxmax()]
max_cant = pdf.loc[pdf["transaction_count"].idxmax()]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(pdf["sale_date"], pdf["total_amount"], marker="o", color="#2563eb")
ax1.scatter([max_monto["sale_date"]], [max_monto["total_amount"]],
            color="#dc2626", s=90, zorder=3, label="Día de mayor monto")
ax1.set_title("Monto vendido y cantidad de transacciones por día")
ax1.set_ylabel("Monto total")
ax1.legend()
ax1.grid(alpha=0.25)

ax2.bar(pdf["sale_date"], pdf["transaction_count"], color="#94a3b8")
ax2.bar([max_cant["sale_date"]], [max_cant["transaction_count"]],
        color="#dc2626", hatch="//", label="Día con más transacciones")
ax2.set_ylabel("Transacciones (cantidad)")
ax2.set_xlabel("Fecha")
ax2.legend()
ax2.grid(axis="y", alpha=0.25)

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

print(f"Mayor monto: {max_monto['sale_date']} ({max_monto['total_amount']:,.2f})")
print(f"Más transacciones: {max_cant['sale_date']} ({int(max_cant['transaction_count']):,})")
print("¿Coinciden los dos días?:", max_monto["sale_date"] == max_cant["sale_date"])

pdf["ticket_promedio"] = pdf["total_amount"] / pdf["transaction_count"]
display(spark.createDataFrame(pdf[["sale_date", "total_amount", "transaction_count", "ticket_promedio"]]))

print(f"Ticket promedio del día de mayor monto: {max_monto['total_amount'] / max_monto['transaction_count']:,.2f}")
print(f"Ticket promedio general: {pdf['total_amount'].sum() / pdf['transaction_count'].sum():,.2f}")

Las salidas nos muestran que en ambos casos el dia de mayor facturacion coincide con el de mas trasacciones. (01/03/2026). A su vez el ticket promedio nos brinda una informacion adicional, y es que gracias a que fue el dia de mayor cantidad de transacciones se logró ser el dia de mayor facturación, ya que el dia de mayor ticket promedio fue el 02/03/2026 (teniendo un 0,2% de mayor facturacion en promedio)

## Consigna 2 — Canal y fraude

**Pregunta:** ¿Qué canal de pago presenta la mayor tasa de fraude? ¿La conclusión se sostiene al considerar el número de transacciones de cada canal?

Mostrá tasa y volumen de manera legible. No compares sólo cantidades de fraude: calculá el denominador correcto. Fuente sugerida: `gold_daily_sales`.

In [0]:
canal = (spark.table(daily)
    .groupBy("payment_channel")
    .agg(F.sum("fraud_transactions").alias("fraudes"),
         F.sum("transaction_count").alias("transacciones"))
    .withColumn("tasa_fraude", F.col("fraudes") / F.col("transacciones"))
    .orderBy(F.desc("tasa_fraude")))
display(canal)

pdf = canal.toPandas().sort_values("tasa_fraude")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

ax1.barh(pdf["payment_channel"], pdf["tasa_fraude"] * 100, color="#dc2626")
for y, v in zip(pdf["payment_channel"], pdf["tasa_fraude"] * 100):
    ax1.text(v, y, f" {v:.2f}%", va="center")
ax1.set_title("Tasa de fraude por canal")
ax1.set_xlabel("Tasa de fraude (%)")
ax1.set_ylabel("Canal de pago")
ax1.grid(axis="x", alpha=0.25)

ax2.barh(pdf["payment_channel"], pdf["transacciones"], color="#64748b")
for y, v in zip(pdf["payment_channel"], pdf["transacciones"]):
    ax2.text(v, y, f" {v:,}", va="center")
ax2.set_title("Volumen de transacciones por canal")
ax2.set_xlabel("Transacciones (cantidad)")
ax2.grid(axis="x", alpha=0.25)

fig.tight_layout()
plt.show()

El dato crudo nos muestra que las transferencias es el canal donde encontramos mas transacciones fraudulentas (2.327); a su vez si bien es el canal con menor cantidad de transacciones (16.723), tan solo es menor en 0,006% frente a wallet y 1,07% a tarjeta. En la definicion de tasa de fraude por canal vemos que transferencias muestra el mayor porcentaje con casi un 14%.

## Consigna 3 — Concentración geográfica y de producto

**Pregunta:** ¿Qué combinación de país y categoría genera el mayor monto? ¿Existe una categoría dominante en todos los países o cambia según el mercado?

Elegí una visualización que permita comparar simultáneamente países y categorías. Justificá brevemente por qué ese tipo de gráfico es adecuado. Fuente sugerida: `gold_daily_sales`.

In [0]:
from pyspark.sql.window import Window

geo = (spark.table(daily)
    .groupBy("country", "category")
    .agg(F.sum("total_amount").cast("double").alias("total_amount")))

pdf = (geo.toPandas()
    .pivot(index="country", columns="category", values="total_amount")
    .fillna(0))

# Ordenar países y categorías por monto total (de mayor a menor)
pdf = pdf.loc[pdf.sum(axis=1).sort_values(ascending=False).index,
              pdf.sum(axis=0).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(pdf.values / 1e6, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(pdf.columns)))
ax.set_xticklabels(pdf.columns, rotation=30, ha="right")
ax.set_yticks(range(len(pdf.index)))
ax.set_yticklabels(pdf.index)
for i in range(len(pdf.index)):
    for j in range(len(pdf.columns)):
        ax.text(j, i, f"{pdf.values[i, j] / 1e6:.2f}", ha="center", va="center", fontsize=8,
                color="white" if pdf.values[i, j] > pdf.values.mean() else "black")
ax.set_title("Monto vendido por país y categoría (millones)")
ax.set_xlabel("Categoría")
ax.set_ylabel("País")
fig.colorbar(im, ax=ax, label="Monto total (millones)")
fig.tight_layout()
plt.show()

top = geo.orderBy(F.desc("total_amount")).first()
print(f"Combinación líder: {top.country} - {top.category} ({top.total_amount:,.2f})")

w = Window.partitionBy("country").orderBy(F.desc("total_amount"))
display(geo.withColumn("puesto", F.row_number().over(w))
           .where("puesto = 1")
           .orderBy(F.desc("total_amount")))


El grafico nos muestra que el mayor volumen facturado lo tenemos en Brasil en la division de "home". Si bien en cada pais este sector tiene una facturacion proponderante en el general, no es el mayor en todos los casos. En Argentina y Mexico observamos que la division "books" es la que genera un mayor nivel de facturacion. 

El formato del grafico me parece adecuado ya que nos permite ubicar a simple vista cuales son las combinaciones de paises y divisiones que generan mayor nivel de facturacion y aquellas otras que lo hacen en niveles inferiores. A su vez ajustamos poniendo los paises que mas facturan arriba y los que menos abajo, y las divisiones que mas lo hacen a la izquieras y a la derecha la que menos. Una nota rapidamente cual es, por ejemplo la division y pais que genera una mayor facturacion debido al color azul oscuro que produce, y a la inversa en los colores claros.

## Consigna 4 — Calidad del pipeline

**Pregunta:** ¿Qué proporción de cada lote fue aceptada y rechazada? ¿El lote nuevo presenta una calidad diferente del lote inicial?

Compará lotes usando proporciones además de cantidades absolutas. Señalá cualquier limitación de comparar lotes de tamaños distintos. Fuente sugerida: `gold_batch_summary`.

In [0]:
lotes = (spark.table(batch)
    .withColumn("total", F.col("accepted_transactions") + F.col("rejected_transactions"))
    .withColumn("pct_aceptadas", F.col("accepted_transactions") / F.col("total"))
    .withColumn("pct_rechazadas", F.col("rejected_transactions") / F.col("total"))
    .orderBy("source_batch_id"))
display(lotes.select("source_batch_id", "accepted_transactions", "rejected_transactions",
                     "total", "pct_aceptadas", "pct_rechazadas"))

pdf = lotes.toPandas()
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.barh(pdf["source_batch_id"], pdf["pct_aceptadas"] * 100,
        color="#16a34a", label="Aceptadas")
ax.barh(pdf["source_batch_id"], pdf["pct_rechazadas"] * 100,
        left=pdf["pct_aceptadas"] * 100, color="#dc2626", hatch="//", label="Rechazadas")
for y, r, p, t in zip(pdf["source_batch_id"], pdf["rejected_transactions"],
                      pdf["pct_rechazadas"] * 100, pdf["total"]):
    ax.text(50, y, f"{r} rechazadas de {t:,} ({p:.2f}%)", ha="center", va="center", color="white")
ax.set_xlim(0, 100)
ax.set_title("Proporción de transacciones aceptadas y rechazadas por lote")
ax.set_xlabel("Porcentaje del lote (%)")
ax.set_ylabel("Lote")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

Podemos observar que en terminos totales hemos tenido 55 rechazos, siendo 51 del inicial y 4 en los nuevos lotes (2 en cada uno). Si lo planteamos en terminos porcentuales podemos decir que los nuevos lotes tienen practicamente 10 veces mas rechazo que el inicial y deberiamos contemplar algunanalisis de estos rechazos, pero a su vez, si lo vemos en valores totales, al ser una muestra radicalmente mas chica nos cesga el resultado porcentual (1 solo rechazo en los nuevos lotes son 5 veces en proporcion al inicial). Concluyo que hay que analisir el global en todas las situaciones, no siempre enfocarse en la tasa sino tambien a veces en los valores absolutos. 

## Criterios de entrega

Para cada consigna entregá: (1) la transformación de datos, (2) un gráfico, y (3) una conclusión de dos o tres oraciones que responda explícitamente la pregunta.

Cada gráfico debe tener título informativo, ejes y unidades legibles, categorías ordenadas cuando corresponda y una elección de color que no sea la única forma de comunicar el significado. No conviertas una tabla Silver completa a Pandas: agregá primero con Spark y convertí sólo el resultado pequeño.